In [17]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2

# Configuration
SEED = 42
IMG_SIZE = 224
SHAP_VALUES_PATH = "outputs/shap_montgomery/shap_values.npy"
SHAP_METADATA_PATH = "outputs/shap_montgomery/shap_metadata.csv"
MONTGOMERY_DIR = Path("data/Montgomery")
OUTPUT_DIR = Path("outputs/lung_relevance")
PLOT_DIR = OUTPUT_DIR / "plots"
OVERLAY_DIR = OUTPUT_DIR / "lung_mask_overlays"
EXAMPLE_DIR = OUTPUT_DIR / "example_panels"

for folder in [OUTPUT_DIR, PLOT_DIR, OVERLAY_DIR, EXAMPLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print("Current working directory:", os.getcwd())
print("Montgomery dir:", MONTGOMERY_DIR.resolve())

Current working directory: /user/HS401/mn01409/Documents/Dissertation
Montgomery dir: /user/HS401/mn01409/Documents/Dissertation/data/Montgomery


In [18]:
# Helper Functions
def normalise_shap_values(shap_values):
    """
    Ensures SHAP shape is N x C x H x W.
    """
    shap_values = np.asarray(shap_values)

    # Sometimes output is N x C x H x W x 1
    if shap_values.ndim == 5 and shap_values.shape[-1] == 1:
        shap_values = shap_values[..., 0]

    # Sometimes output is N x H x W x C
    if shap_values.ndim == 4 and shap_values.shape[-1] == 3:
        shap_values = np.transpose(shap_values, (0, 3, 1, 2))

    if shap_values.ndim != 4:
        raise ValueError(f"Unexpected SHAP values shape: {shap_values.shape}")

    if shap_values.shape[1] != 3:
        raise ValueError(f"Expected N x C x H x W, got {shap_values.shape}")

    return shap_values

def shap_to_heatmap(shap_value):
    """
    Convert SHAP value to a positive 2D attribution heatmap.
    Input expected: C x H x W.
    """
    shap_value = np.asarray(shap_value)

    if shap_value.ndim != 3:
        raise ValueError(f"Expected 3D SHAP value, got {shap_value.shape}")

    if shap_value.shape[0] == 3:
        heatmap = np.abs(shap_value).mean(axis=0)
    elif shap_value.shape[-1] == 3:
        heatmap = np.abs(shap_value).mean(axis=-1)
    else:
        raise ValueError(f"Unexpected SHAP shape: {shap_value.shape}")

    return heatmap.astype(np.float32)

def normalise_heatmap_for_display(heatmap):
    heatmap = np.asarray(heatmap).astype(np.float32)
    heatmap = heatmap - heatmap.min()
    heatmap = heatmap / (heatmap.max() + 1e-8)
    return heatmap

def clean_image_id(image_id):
    """
    Normalise image IDs so both of these work:
    MCUCXR_0255_1
    MCUCXR_0255_1.png
    """
    return Path(str(image_id)).stem

def locate_mask_dirs(montgomery_dir, metadata_df=None):
    """
    Robustly find ManualMask/leftMask and ManualMask/rightMask.

    Handles:
    data/Montgomery/ManualMask/leftMask
    data/Montgomery/MontgomerySet/ManualMask/leftMask
    or any ManualMask folder under data/Montgomery.
    """
    montgomery_dir = Path(montgomery_dir)

    candidate_roots = [
        montgomery_dir,
        montgomery_dir / "MontgomerySet",
        montgomery_dir.parent / "MontgomerySet",
    ]

    # Also search upward from the first SHAP image path
    if metadata_df is not None and len(metadata_df) > 0 and "image_path" in metadata_df.columns:
        first_image_path = Path(str(metadata_df.iloc[0]["image_path"]))
        for parent in first_image_path.parents:
            candidate_roots.append(parent)
            candidate_roots.append(parent.parent)

    checked = []

    for root in candidate_roots:
        root = Path(root)
        manual = root / "ManualMask"
        left = manual / "leftMask"
        right = manual / "rightMask"

        checked.append(str(manual))

        if left.exists() and right.exists():
            return left, right

    # Fallback: recursive search under montgomery_dir
    for manual in montgomery_dir.rglob("ManualMask"):
        left = manual / "leftMask"
        right = manual / "rightMask"
        checked.append(str(manual))
        if left.exists() and right.exists():
            return left, right

    raise FileNotFoundError(
        "Could not find ManualMask/leftMask and ManualMask/rightMask. "
        "Checked these locations:\n" + "\n".join(checked)
    )

def find_mask_file(mask_dir, image_id):
    """
    Find matching mask file using robust filename matching.
    Official masks are usually named like MCUCXR_0255_1.png.
    """
    mask_dir = Path(mask_dir)
    image_stem = clean_image_id(image_id)

    if not mask_dir.exists():
        return None

    possible_extensions = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

    # Direct match
    for ext in possible_extensions:
        candidate = mask_dir / f"{image_stem}{ext}"
        if candidate.exists():
            return str(candidate)

    # Case-insensitive exact stem match
    for file in mask_dir.iterdir():
        if file.is_file() and file.stem.lower() == image_stem.lower():
            return str(file)

    # Fallback for possible suffix mismatch
    # Example: image ID may include _0/_1 while another file does not, or vice versa
    short_stem = "_".join(image_stem.split("_")[:-1]) if "_" in image_stem else image_stem

    for file in mask_dir.iterdir():
        if not file.is_file():
            continue
        file_stem = file.stem.lower()
        if file_stem == short_stem.lower():
            return str(file)
        if image_stem.lower() in file_stem or file_stem in image_stem.lower():
            return str(file)

    return None

def load_lung_mask(image_id):
    """
    Load and combine Montgomery left and right lung masks.
    Returns binary mask resized to IMG_SIZE x IMG_SIZE.
    """
    left_mask_path = find_mask_file(LEFT_MASK_DIR, image_id)
    right_mask_path = find_mask_file(RIGHT_MASK_DIR, image_id)

    if left_mask_path is None or right_mask_path is None:
        return None, left_mask_path, right_mask_path

    left_mask = Image.open(left_mask_path).convert("L").resize((IMG_SIZE, IMG_SIZE))
    right_mask = Image.open(right_mask_path).convert("L").resize((IMG_SIZE, IMG_SIZE))

    left_np = np.array(left_mask).astype(np.float32)
    right_np = np.array(right_mask).astype(np.float32)

    binary_mask = ((left_np + right_np) > 0).astype(np.uint8)
    return binary_mask, left_mask_path, right_mask_path

def load_display_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = image.resize((IMG_SIZE, IMG_SIZE))
    image_np = np.array(image).astype(np.float32) / 255.0
    return image_np

def calculate_lung_relevance_metrics(heatmap, lung_mask):
    """
    heatmap: positive SHAP attribution, H x W
    lung_mask: binary mask, H x W
    """
    heatmap = np.asarray(heatmap).astype(np.float32)
    lung_mask = np.asarray(lung_mask).astype(np.uint8)

    total_attr = heatmap.sum() + 1e-8
    inside_attr = (heatmap * lung_mask).sum()
    outside_attr = (heatmap * (1 - lung_mask)).sum()

    lung_attribution_ratio = inside_attr / total_attr
    outside_attribution_ratio = outside_attr / total_attr

    lung_area_fraction = lung_mask.mean()
    outside_area_fraction = 1.0 - lung_area_fraction

    lung_enrichment = lung_attribution_ratio / (lung_area_fraction + 1e-8)
    outside_enrichment = outside_attribution_ratio / (outside_area_fraction + 1e-8)

    return {
        "total_attribution": float(total_attr),
        "inside_lung_attribution": float(inside_attr),
        "outside_lung_attribution": float(outside_attr),
        "lung_attribution_ratio": float(lung_attribution_ratio),
        "outside_attribution_ratio": float(outside_attribution_ratio),
        "lung_area_fraction": float(lung_area_fraction),
        "outside_area_fraction": float(outside_area_fraction),
        "lung_enrichment": float(lung_enrichment),
        "outside_enrichment": float(outside_enrichment)
    }

def create_overlay(image_np, heatmap, lung_mask):
    heatmap_display = normalise_heatmap_for_display(heatmap)

    heatmap_uint8 = np.uint8(255 * heatmap_display)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0

    shap_overlay = 0.55 * image_np + 0.45 * heatmap_color
    shap_overlay = np.clip(shap_overlay, 0, 1)

    mask_boundary = cv2.Canny((lung_mask * 255).astype(np.uint8), 50, 150)
    boundary_rgb = np.zeros_like(image_np)
    boundary_rgb[:, :, 1] = mask_boundary / 255.0

    mask_overlay = np.clip(image_np + boundary_rgb, 0, 1)
    combined_overlay = np.clip(shap_overlay + boundary_rgb * 0.8, 0, 1)

    return heatmap_display, shap_overlay, mask_overlay, combined_overlay


def save_lung_relevance_panel(image_np, heatmap, lung_mask, save_path, title):
    heatmap_display, shap_overlay, mask_overlay, combined_overlay = create_overlay(
        image_np, heatmap, lung_mask
    )

    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image_np)
    plt.axis("off")
    plt.title("Original X-ray")

    plt.subplot(1, 4, 2)
    plt.imshow(lung_mask, cmap="gray")
    plt.axis("off")
    plt.title("Lung mask")

    plt.subplot(1, 4, 3)
    plt.imshow(heatmap_display, cmap="hot")
    plt.axis("off")
    plt.title("SHAP heatmap")

    plt.subplot(1, 4, 4)
    plt.imshow(combined_overlay)
    plt.axis("off")
    plt.title("SHAP + lung boundary")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

In [19]:
# Load SHAP, metadata, and locate masks
shap_values = np.load(SHAP_VALUES_PATH)
shap_values = normalise_shap_values(shap_values)
meta_df = pd.read_csv(SHAP_METADATA_PATH)

print("SHAP values shape:", shap_values.shape)
print("Metadata shape:", meta_df.shape)

if len(meta_df) != shap_values.shape[0]:
    raise ValueError(
        f"Metadata rows ({len(meta_df)}) do not match SHAP values ({shap_values.shape[0]}). "
        "Use the matching shap_values.npy and shap_metadata.csv from the same SHAP run."
    )

LEFT_MASK_DIR, RIGHT_MASK_DIR = locate_mask_dirs(MONTGOMERY_DIR, meta_df)

print("Left mask dir:", LEFT_MASK_DIR)
print("Right mask dir:", RIGHT_MASK_DIR)
print("Left mask examples:", sorted(os.listdir(LEFT_MASK_DIR))[:5])
print("Right mask examples:", sorted(os.listdir(RIGHT_MASK_DIR))[:5])

# Verifying first 10 mask matches before full run
print("\nChecking first 10 mask matches:")
for image_id in meta_df["image_id"].astype(str).head(10):
    lung_mask, left_path, right_path = load_lung_mask(image_id)
    print(image_id, "->", lung_mask is not None, "| left:", left_path, "| right:", right_path)

# Lung Relevance Analysis
results = []
missing_masks = []
for idx in range(len(meta_df)):
    row = meta_df.iloc[idx]

    image_id = str(row["image_id"])
    image_path = row["image_path"]

    lung_mask, left_mask_path, right_mask_path = load_lung_mask(image_id)

    if lung_mask is None:
        missing_masks.append({
            "image_id": image_id,
            "image_path": image_path,
            "left_mask_found": left_mask_path is not None,
            "right_mask_found": right_mask_path is not None
        })
        continue

    heatmap = shap_to_heatmap(shap_values[idx])
    metrics = calculate_lung_relevance_metrics(heatmap, lung_mask)

    result = {
        "image_id": image_id,
        "image_path": image_path,
        "true_label": row.get("true_label", np.nan),
        "cnn_prediction": row.get("cnn_prediction", np.nan),
        "cnn_probability_tb": row.get("cnn_probability_tb", np.nan),
        "case_type": row.get("case_type", "Unknown"),
        "left_mask_path": left_mask_path,
        "right_mask_path": right_mask_path
    }

    if "dataset_name" in meta_df.columns:
        result["dataset_name"] = row.get("dataset_name", "Montgomery")

    result.update(metrics)
    results.append(result)

results_df = pd.DataFrame(results)
missing_df = pd.DataFrame(missing_masks)

results_path = OUTPUT_DIR / "lung_relevance_results.csv"
missing_path = OUTPUT_DIR / "missing_lung_masks.csv"

results_df.to_csv(results_path, index=False)
missing_df.to_csv(missing_path, index=False)

print(f"Saved lung relevance results to {results_path}")
print(f"Saved missing mask list to {missing_path}")
print(f"Images with masks analysed: {len(results_df)}")
print(f"Images missing masks: {len(missing_df)}")

if len(results_df) == 0:
    raise ValueError(
        "No lung masks were matched. The mask folders exist, but filenames are not matching image_id. "
        "Open missing_lung_masks.csv and compare image_id with files in ManualMask/leftMask."
    )

# Summary Statistics
summary = {
    "n_images": len(results_df),
    "mean_lung_attribution_ratio": results_df["lung_attribution_ratio"].mean(),
    "std_lung_attribution_ratio": results_df["lung_attribution_ratio"].std(),
    "median_lung_attribution_ratio": results_df["lung_attribution_ratio"].median(),
    "mean_outside_attribution_ratio": results_df["outside_attribution_ratio"].mean(),
    "std_outside_attribution_ratio": results_df["outside_attribution_ratio"].std(),
    "mean_lung_area_fraction": results_df["lung_area_fraction"].mean(),
    "std_lung_area_fraction": results_df["lung_area_fraction"].std(),
    "mean_lung_enrichment": results_df["lung_enrichment"].mean(),
    "std_lung_enrichment": results_df["lung_enrichment"].std()
}

summary_df = pd.DataFrame([summary])
summary_path = OUTPUT_DIR / "lung_relevance_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nLung relevance summary:")
display(summary_df)

# Case-Type Summary
case_summary = results_df.groupby("case_type").agg(
    n=("image_id", "count"),
    mean_lung_attribution_ratio=("lung_attribution_ratio", "mean"),
    std_lung_attribution_ratio=("lung_attribution_ratio", "std"),
    mean_lung_enrichment=("lung_enrichment", "mean"),
    mean_cnn_probability_tb=("cnn_probability_tb", "mean")
).reset_index()

case_summary_path = OUTPUT_DIR / "lung_relevance_by_case_type.csv"
case_summary.to_csv(case_summary_path, index=False)

print("\nCase-type summary:")
display(case_summary)

SHAP values shape: (138, 3, 224, 224)
Metadata shape: (138, 7)
Left mask dir: data/Montgomery/ManualMask/leftMask
Right mask dir: data/Montgomery/ManualMask/rightMask
Left mask examples: ['MCUCXR_0001_0.png', 'MCUCXR_0002_0.png', 'MCUCXR_0003_0.png', 'MCUCXR_0004_0.png', 'MCUCXR_0005_0.png']
Right mask examples: ['MCUCXR_0001_0.png', 'MCUCXR_0002_0.png', 'MCUCXR_0003_0.png', 'MCUCXR_0004_0.png', 'MCUCXR_0005_0.png']

Checking first 10 mask matches:
MCUCXR_0255_1 -> True | left: data/Montgomery/ManualMask/leftMask/MCUCXR_0255_1.png | right: data/Montgomery/ManualMask/rightMask/MCUCXR_0255_1.png
MCUCXR_0309_1 -> True | left: data/Montgomery/ManualMask/leftMask/MCUCXR_0309_1.png | right: data/Montgomery/ManualMask/rightMask/MCUCXR_0309_1.png
MCUCXR_0173_1 -> True | left: data/Montgomery/ManualMask/leftMask/MCUCXR_0173_1.png | right: data/Montgomery/ManualMask/rightMask/MCUCXR_0173_1.png
MCUCXR_0059_0 -> True | left: data/Montgomery/ManualMask/leftMask/MCUCXR_0059_0.png | right: data/Montg

,n_images,mean_lung_attribution_ratio,std_lung_attribution_ratio,median_lung_attribution_ratio,mean_outside_attribution_ratio,std_outside_attribution_ratio,mean_lung_area_fraction,std_lung_area_fraction,mean_lung_enrichment,std_lung_enrichment
0,138,0.415302,0.106852,0.421406,0.584698,0.106852,0.265587,0.062922,1.560216,0.15987



Case-type summary:


,case_type,n,mean_lung_attribution_ratio,std_lung_attribution_ratio,mean_lung_enrichment,mean_cnn_probability_tb
0,FP,3,0.505397,0.070855,1.598089,0.832414
1,TN,77,0.400806,0.109791,1.572600,0.026658
2,TP,58,0.429886,0.101322,1.541816,0.990535


In [20]:
# Visualisations

# Plot 1: Lung Attribution Ratio Histogram
plt.figure(figsize=(8, 5))
plt.hist(results_df["lung_attribution_ratio"], bins=15, edgecolor="black")
plt.xlabel("Lung attribution ratio")
plt.ylabel("Number of images")
plt.title("Distribution of SHAP Attribution Inside Lung Region")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lung_attribution_ratio_histogram.png", dpi=300)
plt.close()

# Plot 2: Lung Attribution Ratio by Case Type
case_order = ["TP", "TN", "FP", "FN"]
available_cases = [c for c in case_order if c in results_df["case_type"].unique()]

if len(available_cases) > 0:
    data_to_plot = [
        results_df[results_df["case_type"] == c]["lung_attribution_ratio"]
        for c in available_cases
    ]

    plt.figure(figsize=(8, 5))
    plt.boxplot(data_to_plot, labels=available_cases)
    plt.ylabel("Lung attribution ratio")
    plt.xlabel("Case type")
    plt.title("Lung Attribution Ratio by Prediction Case Type")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "lung_attribution_ratio_by_case_type.png", dpi=300)
    plt.close()

# Plot 3: Lung Attribution vs Lung Area
plt.figure(figsize=(6, 6))
plt.scatter(
    results_df["lung_area_fraction"],
    results_df["lung_attribution_ratio"],
    alpha=0.75
)

min_val = min(
    results_df["lung_area_fraction"].min(),
    results_df["lung_attribution_ratio"].min()
)
max_val = max(
    results_df["lung_area_fraction"].max(),
    results_df["lung_attribution_ratio"].max()
)

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Lung area fraction")
plt.ylabel("Lung attribution ratio")
plt.title("Lung Attribution Compared with Lung Area")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lung_attribution_vs_lung_area.png", dpi=300)
plt.close()

# Plot 4: Lung Enrichment
plt.figure(figsize=(8, 5))
plt.hist(results_df["lung_enrichment"], bins=15, edgecolor="black")
plt.axvline(1.0, linestyle="--")
plt.xlabel("Lung enrichment")
plt.ylabel("Number of images")
plt.title("SHAP Lung Enrichment Distribution")
plt.tight_layout()
plt.savefig(PLOT_DIR / "lung_enrichment_histogram.png", dpi=300)
plt.close()

print("Saved plots to:", PLOT_DIR)

# Save Example Panels
examples_saved = 0
max_examples = 8

selected_indices = []

# Prefer one example per case type first
for case_type in ["TP", "TN", "FP", "FN"]:
    case_rows = results_df[results_df["case_type"] == case_type]
    if len(case_rows) > 0:
        selected_indices.append(case_rows.index[0])

# Add high lung-ratio examples
if len(selected_indices) < max_examples:
    remaining = results_df.sort_values(
        "lung_attribution_ratio",
        ascending=False
    ).index.tolist()

    for idx in remaining:
        if idx not in selected_indices:
            selected_indices.append(idx)
        if len(selected_indices) >= max_examples:
            break

for idx in selected_indices[:max_examples]:
    row = results_df.loc[idx]

    image_id = str(row["image_id"])
    image_path = row["image_path"]

    # Match the same SHAP row by image_id
    original_meta_idx = meta_df[meta_df["image_id"].astype(str) == image_id].index[0]

    image_np = load_display_image(image_path)
    heatmap = shap_to_heatmap(shap_values[original_meta_idx])
    lung_mask, _, _ = load_lung_mask(image_id)

    title = (
        f"ID: {image_id} | Case: {row['case_type']} | "
        f"Lung ratio: {row['lung_attribution_ratio']:.3f} | "
        f"Enrichment: {row['lung_enrichment']:.3f}"
    )

    save_path = EXAMPLE_DIR / f"{clean_image_id(image_id)}_lung_relevance_panel.png"

    save_lung_relevance_panel(
        image_np,
        heatmap,
        lung_mask,
        save_path,
        title
    )

    examples_saved += 1

print(f"Saved {examples_saved} lung relevance example panels to {EXAMPLE_DIR}.")

/tmp/ipykernel_121011/678887321.py:24: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_to_plot, labels=available_cases)


Saved plots to: outputs/lung_relevance/plots
Saved 8 lung relevance example panels to outputs/lung_relevance/example_panels.


In [21]:
print("Output files:")
for file in sorted(OUTPUT_DIR.rglob("*")):
    if file.is_file():
        print(file)

Output files:
outputs/lung_relevance/example_panels/MCUCXR_0022_0_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0051_0_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0059_0_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0069_0_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0079_0_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0099_0_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0141_1_lung_relevance_panel.png
outputs/lung_relevance/example_panels/MCUCXR_0255_1_lung_relevance_panel.png
outputs/lung_relevance/lung_relevance_by_case_type.csv
outputs/lung_relevance/lung_relevance_results.csv
outputs/lung_relevance/lung_relevance_summary.csv
outputs/lung_relevance/missing_lung_masks.csv
outputs/lung_relevance/plots/lung_attribution_ratio_by_case_type.png
outputs/lung_relevance/plots/lung_attribution_ratio_histogram.png
outputs/lung_relevance/plots/lung_